<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 00 Environment

The notebook checks the tools every other notebook relies on: the pinned toolchain with the target of
the Web Worker, wasm-bindgen of the version the lock of the worker pins, the submodule of Aosabi, the
devtool, the built module of WebAssembly and the built GraphQL engine. It fails first, so a missing
tool reads as such and not as a broken feature.

In [ ]:
require_build = True

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

## 1. The toolchain

In [ ]:
import re, subprocess
pin = re.search(r'channel = "([^"]+)"', (ROOT / 'rust-toolchain.toml').read_text()).group(1)
rustc = subprocess.run(['rustc', '--version'], cwd=ROOT, capture_output=True, text=True).stdout
check(pin in rustc, f'rustc is the pinned {pin}', rustc)
targets = subprocess.run(['rustup', 'target', 'list', '--installed'], cwd=ROOT, capture_output=True, text=True).stdout
check('wasm32-unknown-unknown' in targets, 'the target wasm32-unknown-unknown is installed', targets)

## 2. wasm-bindgen and the lock

In [ ]:
lock = (ROOT / 'crates' / 'bausteine-web' / 'Cargo.lock').read_text()
pinned = re.search(r'name = "wasm-bindgen"\nversion = "([^"]+)"', lock).group(1)
cli = subprocess.run(['wasm-bindgen', '--version'], capture_output=True, text=True).stdout
check(pinned in cli, f'wasm-bindgen-cli is the locked {pinned}', cli)

## 3. The submodule of Aosabi

In [ ]:
facade = ROOT / 'third_party' / 'aosabi' / 'crates' / 'aosabi' / 'Cargo.toml'
check(facade.exists(), 'the submodule third_party/aosabi is checked out')

## 4. The devtool and the builds

In [ ]:
build_devtool()
check(len(devtool('examples')) == 5, 'the devtool answers the five examples')
if require_build:
    check((ROOT / 'web' / 'pkg' / 'bausteine_web_bg.wasm').exists(), 'the module of WebAssembly is built into web/pkg')
    check(ENGINE.exists(), 'the GraphQL engine bausteine-graphql is built')
    check((ROOT / 'web' / 'js' / 'worker.js').exists(), 'the script of the Web Worker is there')